In [3]:
### RAG pipeline -Data ingestion to vector DB pipeline

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 


 

In [28]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files ina a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory) 
    
    #find all the pdf files recursively 
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print (f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\n Processing :{pdf_file.name }")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            
            #add source information to metadata 
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
                
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        
        except Exception as e:
            print (f" Error {e}")
            
    
    print (f"\n Total documents loaded :{len(all_documents)}")
    return all_documents 
            
            
#process all pdfs in the data directory
all_pdf_documents=process_all_pdfs("../data")
                
    
    
     

found 3 PDF files to process

 Processing :attention.pdf
 Loaded 15 pages

 Processing :Evaluation.pdf
 Loaded 26 pages

 Processing :Jev.pdf
 Loaded 8 pages

 Total documents loaded :49


In [29]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toron

In [30]:
##text splitting get into chunks 

def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance """
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")
    
    
    #example of a chunk 
    
    if split_docs:
        print(f"\n example chunk")
        print(f"Content : {split_docs[0].page_content[:200]}..")
        print(f"metadata: {split_docs[0].metadata}")
    
    return split_docs
    

In [31]:
chunks =split_documents(all_pdf_documents)

split 49 documents into 176 chunks

 example chunk
Content : Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
..
metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


In [32]:
###embedding and vector storeDB

import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity 


In [33]:
class EmbeddingManager:
    """handles document embedding generation using Sentence TRansformer """
    
    def __init__(self,model_name: str= "all-MiniLM-L6-v2"):
        """Iniitialize the embedding manager 
        args: model_name:huggingFace model name for sentence embedding 
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully .Embedding dimension :{self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise 
        
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts 
        args: texts: List of text strings to embed
        Returns: numpy array of embeddings with shape (len(texts),embedding_dim)
         
        """
        
        if not self.model:
            raise ValueError("Model not loaded ")
        
        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape :{embeddings.shape}")
        return embeddings 
            
    # def get_embedding_dimension(self) ->int:
    #     """get the embedding dimension of the model  """
    #     if not self.model:
    #         raise ValueError("model not loaded")
    #     return self.model.get_sentence_embedding_dimension()
    
##initialize embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

        
         

Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6630.95it/s]


Model loaded successfully .Embedding dimension :384


In [34]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf-documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize ChromaDB client and collection.
        """
        try:
            # Create persistent Chroma client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create VectorStore object
vectorstore = VectorStore()


Vector store initialized. Collection: pdf-documents
Existing documents in collection: 0


In [35]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toron

In [36]:
###convert text to embeddings

texts=[doc.page_content for doc in chunks ]


### generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store in the vectorstore

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 176 texts ...


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.50it/s]

Generated embeddings with shape :(176, 384)
Adding 176 documents to vector store...
Successfully added 176 documents to vector store
Total documents in collection: 176


###Retriever pipeline from vectoStore 

In [37]:
class RAGRetriever:
    """Handles query based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity threshold.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            print("DOCUMENTS:", results["documents"])
            print("DISTANCES:", results["distances"])
            print("IDS:", results["ids"])

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [41]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]

Generated embeddings with shape :(1, 384)
DOCUMENTS: [['3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3', 'itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset

[{'id': 'doc_d7ac26db',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'title': '',
   'author': '',
   'keywords': '',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'content_length': 216,
   'source': '../data/pdf/attention.pdf',
   'source_file': 'attention.pdf',
   'page': 2,
   'moddate': '2024-04-10T21:11:43+00:00',
   'doc_index': 12,
   'trapped': '/False',
   'file_type': 'pdf',
   'page_label': '3',
   'subject': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'total_pages': 15,
   'producer': 'pdfTeX-1.40.25'},
  'similarity_score': 0.13995492458343506,
  'distance': 0.8600450754165649,
  'rank': 1}]

###Integration vectorDB context pipeline with LLM output


In [ ]:
###Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


#initialize the Groq LLM (set your GROQ_API_KEY in env)
groq_api_key=os.getenv("GROQ_API_KEY")



llm =ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

#simple RAG function. ->retreive context + generate response 

def rag_simple(query,retriever,llm ,top_k=3):
    #retrieve the context 
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question concisely"
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
    Context:
    {context}
    
    Question:{query}
    
    Answer:"""
    
    response =llm.invoke([prompt.format(context=context,query=query)])
    return response.content 


    
        


In [53]:
answer=rag_simple("what is attention is all you need",rag_retriever,llm)
answer

Retrieving documents for query: 'what is attention is all you need'
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.18it/s]

Generated embeddings with shape :(1, 384)
DOCUMENTS: [['3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3', 'itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset

'**“Attention Is All You Need”** is the 2017 paper that introduced the **Transformer** architecture. It showed that a model built solely on attention mechanisms—specifically self‑attention that maps queries, keys, and values to weighted‑sum outputs—can replace recurrent or convolutional layers and achieve state‑of‑the‑art performance in tasks like machine translation. In short, the paper demonstrates that attention alone is sufficient for effective sequence modeling.'